# Lựa chọn đặc trưng bằng RFE

Notebook này dùng phương pháp Recursive Feature Elimination để tìm số lượng đặc trưng phù hợp cho bài toán dự đoán tuổi Abalone.

## 1. Import thư viện

In [ ]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.feature_selection import RFE
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)
np.random.seed(42)

## 2. Nạp dữ liệu

In [ ]:
duong_dan_ung_vien = [
    Path("../../data/raw/abalone.csv"),
    Path("../data/raw/abalone.csv"),
    Path("data/raw/abalone.csv"),
    Path("AbaloneAge/data/raw/abalone.csv"),
]

duong_dan_du_lieu = None
for p in duong_dan_ung_vien:
    p_day_du = p.resolve()
    if p_day_du.exists():
        duong_dan_du_lieu = p_day_du
        break

if duong_dan_du_lieu is None:
    raise FileNotFoundError(
        "Khong tim thay file abalone.csv. Da thu: "
        + ", ".join(str(p.resolve()) for p in duong_dan_ung_vien)
    )

df = pd.read_csv(duong_dan_du_lieu, header=None)
df.columns = [
    "sex",
    "length",
    "diameter",
    "height",
    "whole_weight",
    "shucked_weight",
    "viscera_weight",
    "shell_weight",
    "rings",
]

print("Duong dan du lieu:", duong_dan_du_lieu)
print("Kich thuoc:", df.shape)
df.head()

## 3. Chia train, validation, test

In [ ]:
X = df.drop(columns=["rings"])
y = df["rings"]

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)

## 4. Tien xu ly

In [ ]:
cot_so = [
    "length",
    "diameter",
    "height",
    "whole_weight",
    "shucked_weight",
    "viscera_weight",
    "shell_weight",
]
cot_hang_muc = ["sex"]

bien_doi_so = Pipeline(steps=[
    ("dien_khuyet", SimpleImputer(strategy="median")),
    ("chuan_hoa", StandardScaler()),
])

bien_doi_hang_muc = Pipeline(steps=[
    ("dien_khuyet", SimpleImputer(strategy="most_frequent")),
    ("one_hot", OneHotEncoder(handle_unknown="ignore")),
])

tien_xu_ly = ColumnTransformer(transformers=[
    ("so", bien_doi_so, cot_so),
    ("hang_muc", bien_doi_hang_muc, cot_hang_muc),
])

X_train_txl = tien_xu_ly.fit_transform(X_train)
X_val_txl = tien_xu_ly.transform(X_val)
X_test_txl = tien_xu_ly.transform(X_test)

ten_dac_trung = tien_xu_ly.get_feature_names_out()
tong_so_dac_trung = len(ten_dac_trung)
print("Tong so dac trung sau tien xu ly:", tong_so_dac_trung)

## 5. Thu nghiem nhieu gia tri so dac trung voi RFE

In [ ]:
# Chon cac moc n_features de tranh chay qua cham
ds_n = sorted(set([3, 5, 7, 9, 10, min(12, tong_so_dac_trung), tong_so_dac_trung]))
ket_qua = []

for n in ds_n:
    bo_chon = RFE(estimator=Ridge(alpha=1.0), n_features_to_select=n, step=1)
    bo_chon.fit(X_train_txl, y_train)

    X_train_rfe = bo_chon.transform(X_train_txl)
    X_val_rfe = bo_chon.transform(X_val_txl)

    mo_hinh = Ridge(alpha=1.0)
    mo_hinh.fit(X_train_rfe, y_train)
    du_doan_val = mo_hinh.predict(X_val_rfe)

    mae = mean_absolute_error(y_val, du_doan_val)
    rmse = np.sqrt(mean_squared_error(y_val, du_doan_val))
    r2 = r2_score(y_val, du_doan_val)

    ket_qua.append({
        "so_dac_trung": int(n),
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
    })

bang_ket_qua = pd.DataFrame(ket_qua).sort_values(by="RMSE")
bang_ket_qua

## 6. Bieu do hieu nang theo so dac trung

In [ ]:
bang_plot = bang_ket_qua.sort_values(by="so_dac_trung")

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(bang_plot["so_dac_trung"], bang_plot["MAE"], marker="o")
axes[0].set_title("MAE theo so dac trung")
axes[0].set_xlabel("So dac trung")
axes[0].set_ylabel("MAE")

axes[1].plot(bang_plot["so_dac_trung"], bang_plot["RMSE"], marker="o", color="tomato")
axes[1].set_title("RMSE theo so dac trung")
axes[1].set_xlabel("So dac trung")
axes[1].set_ylabel("RMSE")

axes[2].plot(bang_plot["so_dac_trung"], bang_plot["R2"], marker="o", color="seagreen")
axes[2].set_title("R2 theo so dac trung")
axes[2].set_xlabel("So dac trung")
axes[2].set_ylabel("R2")

plt.tight_layout()
duong_dan_hinh = Path("../../outputs/figures").resolve()
duong_dan_hinh.mkdir(parents=True, exist_ok=True)
plt.savefig(duong_dan_hinh / "03_feature_selection_rfe_metrics.png", dpi=300, bbox_inches="tight")
plt.show()

print("Da luu hinh: 03_feature_selection_rfe_metrics.png")

## 7. Chon cau hinh tot nhat va xem dac trung duoc giu

In [ ]:
n_tot_nhat = int(bang_ket_qua.iloc[0]["so_dac_trung"])
print("So dac trung tot nhat theo RMSE:", n_tot_nhat)

bo_chon_tot_nhat = RFE(estimator=Ridge(alpha=1.0), n_features_to_select=n_tot_nhat, step=1)
bo_chon_tot_nhat.fit(X_train_txl, y_train)

bo_loc = bo_chon_tot_nhat.support_
xep_hang = bo_chon_tot_nhat.ranking_

bang_xep_hang = pd.DataFrame({
    "dac_trung": ten_dac_trung,
    "duoc_chon": bo_loc,
    "xep_hang_rfe": xep_hang,
}).sort_values(by=["xep_hang_rfe", "dac_trung"])

bang_xep_hang.head(20)

## 8. Danh gia tren tap test

In [ ]:
X_train_tot = bo_chon_tot_nhat.transform(X_train_txl)
X_test_tot = bo_chon_tot_nhat.transform(X_test_txl)

mo_hinh_tot = Ridge(alpha=1.0)
mo_hinh_tot.fit(X_train_tot, y_train)
du_doan_test = mo_hinh_tot.predict(X_test_tot)

mae_test = mean_absolute_error(y_test, du_doan_test)
rmse_test = np.sqrt(mean_squared_error(y_test, du_doan_test))
r2_test = r2_score(y_test, du_doan_test)

print(f"MAE test : {mae_test:.4f}")
print(f"RMSE test: {rmse_test:.4f}")
print(f"R2 test  : {r2_test:.4f}")

## 9. Luu ket qua

In [ ]:
duong_dan_metrics = Path("../../outputs/metrics").resolve()
duong_dan_metrics.mkdir(parents=True, exist_ok=True)

bang_ket_qua.to_csv(duong_dan_metrics / "03_rfe_n_features_comparison.csv", index=False)
bang_xep_hang.to_csv(duong_dan_metrics / "03_rfe_feature_ranking.csv", index=False)

tom_tat = {
    "phuong_phap": "rfe",
    "so_dac_trung_tot_nhat": n_tot_nhat,
    "test": {
        "MAE": mae_test,
        "RMSE": rmse_test,
        "R2": r2_test,
    },
    "top_10": bang_xep_hang.query("duoc_chon == True").head(10).to_dict(orient="records"),
}

with open(duong_dan_metrics / "03_rfe_summary.json", "w", encoding="utf-8") as f:
    json.dump(tom_tat, f, ensure_ascii=False, indent=2)

print("Da luu: 03_rfe_n_features_comparison.csv")
print("Da luu: 03_rfe_feature_ranking.csv")
print("Da luu: 03_rfe_summary.json")